In [ ]:
from pathlib import Path
import logging

import h5py
import numpy as np
import pandas as pd


INPUT_DIR = Path(r"S:\Lab_Member\Tobi\Experiments\Exp9_Social-Stress\Raw Data\Behavior\B4\OFT\SLEAP\animal")
OUTPUT_DIR = INPUT_DIR

PART_NAMES = [
    "nose",
    "leftEar",
    "rightEar",
    "leftFlank",
    "rightFlank",
    "tailbase",
    "spine1",
    "leftArm",
    "rightArm",
    "bodycentre",
    "spine2",
    "tail1",
    "tail2",
    "tailtip",
]

OVERWRITE = True

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)


def interpolate_nan_1d(y: np.ndarray) -> np.ndarray:
    """Linearly interpolate NaNs in one 1D time series.

    Leading and trailing NaNs are filled with the nearest valid value.
    Fully missing series remain NaN.
    """
    y = y.astype(float, copy=True)

    valid = ~np.isnan(y)
    if valid.sum() == 0:
        return y

    if valid.sum() == 1:
        y[:] = y[valid][0]
        return y

    x = np.arange(len(y))
    y[~valid] = np.interp(x[~valid], x[valid], y[valid])
    return y


def fill_missing_over_time(arr: np.ndarray) -> np.ndarray:
    """Interpolate missing values over frame/time axis.

    Expected first dimension = frames.
    Other dimensions are interpolated independently.
    """
    arr = np.asarray(arr, dtype=float)
    original_shape = arr.shape

    flat = arr.reshape(original_shape[0], -1)

    for col in range(flat.shape[1]):
        flat[:, col] = interpolate_nan_1d(flat[:, col])

    return flat.reshape(original_shape)


def load_sleap_tracks(path: Path) -> np.ndarray:
    """Load SLEAP tracks and convert to expected layout.

    SLEAP commonly stores tracks as:
    nodes x coordinates x frames x instances

    After `.T`, this becomes:
    instances x frames x coordinates x nodes

    Your original code assumes after `.T`:
    frames x nodes x coordinates x instances

    So this function checks and normalizes explicitly.
    """
    with h5py.File(path, "r") as f:
        if "tracks" not in f:
            raise KeyError(f"No 'tracks' dataset found in {path.name}")

        raw = f["tracks"][:]

    # Common SLEAP layout: nodes x coords x frames x tracks
    if raw.ndim != 4:
        raise ValueError(f"Expected 4D tracks array, got shape {raw.shape}")

    # Convert to: frames x nodes x coords x animals
    # raw: nodes x coords x frames x animals
    tracks = np.transpose(raw, (2, 0, 1, 3))

    n_frames, n_nodes, n_coords, n_animals = tracks.shape

    if n_nodes != len(PART_NAMES):
        raise ValueError(
            f"{path.name}: expected {len(PART_NAMES)} body parts, found {n_nodes}. "
            f"Track shape after transpose: {tracks.shape}"
        )

    if n_coords < 2:
        raise ValueError(f"{path.name}: expected at least x/y coordinates, found {n_coords}")

    return tracks


def tracks_to_dataframe(tracks: np.ndarray) -> pd.DataFrame:
    """Convert tracks to a flat CSV-friendly DataFrame.

    Output has one row per frame per animal.
    """
    n_frames, n_parts, n_coords, n_animals = tracks.shape

    rows = {
        "frame": np.repeat(np.arange(n_frames), n_animals),
        "animal": np.tile(np.arange(n_animals), n_frames),
    }

    for part_idx, part_name in enumerate(PART_NAMES):
        # Shape: frames x animals
        x = tracks[:, part_idx, 0, :]
        y = tracks[:, part_idx, 1, :]

        rows[f"{part_name}_x"] = x.reshape(-1)
        rows[f"{part_name}_y"] = y.reshape(-1)

    return pd.DataFrame(rows)


def process_file(input_path: Path) -> None:
    output_path = OUTPUT_DIR / f"{input_path.stem}_locs.csv"

    if output_path.exists() and not OVERWRITE:
        logging.info("Skipping existing output: %s", output_path.name)
        return

    tracks = load_sleap_tracks(input_path)
    tracks = fill_missing_over_time(tracks)

    df = tracks_to_dataframe(tracks)
    df.to_csv(output_path, index=False)

    logging.info(
        "Processed %s -> %s | rows=%d, columns=%d",
        input_path.name,
        output_path.name,
        df.shape[0],
        df.shape[1],
    )


def main() -> None:
    h5_files = sorted(INPUT_DIR.glob("*.h5"))

    if not h5_files:
        raise FileNotFoundError(f"No .h5 files found in {INPUT_DIR}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for path in h5_files:
        try:
            process_file(path)
        except Exception as exc:
            logging.error("Failed processing %s: %s", path.name, exc)


if __name__ == "__main__":
    main()